# Video decomposition test

In [10]:
from pathlib import Path
import numpy as np

from wavelet_utils import loadFilterParamDict, downscale_binary_video, compute_and_save_dwt

In [11]:
temppath = r'D:\SynologyDriveSyncedDATA\PROCESSED\Waven'

libpath= Path(temppath) / 'gaborLibrary_5_3_8_5_4_4.npy'
libpath= Path(temppath) / 'gaborLibrary_15_10_8_5_4_4.npy'
libpath= Path(temppath) / 'gaborLibrary_40_26_8_5_4_4.npy'

paramspath = libpath.with_suffix('.json')   

videopath = Path(temppath) / 'zebra_s0_d420.0_fps59.94_RESAMPLED13fps.mp4'

In [12]:
library = np.load(libpath)
print(f"Loaded Gabor filter library from {libpath} with shape {library.shape}")

xs, ys, angles, sizes, freqs, phases, visual_coverage, full_screen_coverage, screen_x, screen_y = loadFilterParamDict(paramspath)

Loaded Gabor filter library from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\gaborLibrary_40_26_8_5_4_4.npy with shape (40, 26, 8, 5, 4, 4, 100, 66)


In [13]:
print(f"full_screen_coverage: {full_screen_coverage}, visual_coverage: {visual_coverage}")

full_screen_coverage: [-90   0 -45  45], visual_coverage: [-90   0 -30  30]


## Downsample video

In [14]:
downsampled_video_path=downscale_binary_video(videopath, full_screen_coverage, visual_coverage, screen_x, screen_y, force=False)

Generating cropped and downsampled binary video...
Output file D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED13fps_downscaled.npy already exists. Skipping generation.


## Display downsampled video result

In [15]:
# Displaying original and downsampled video frames
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from pathlib import Path

input_video_path = Path(videopath)
#output_npy_path 

out_video = np.load(downsampled_video_path, mmap_mode="r")

cap = cv2.VideoCapture(str(input_video_path))
n_frames = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), out_video.shape[0])

def show_frame(frame_idx):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        print(f"Could not read frame {frame_idx}")
        return

    input_gray = frame[:, :, 0]
    output_frame = out_video[frame_idx].T

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(input_gray, cmap="gray")
    axes[0].set_title(f"Input frame {frame_idx}")
    axes[0].axis("off")

    axes[1].imshow(output_frame, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(f"Output frame {frame_idx}")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

interact(
    show_frame,
    frame_idx=IntSlider(
        value=0,
        min=0,
        max=n_frames - 1,
        step=1,
        description="Frame"
    )
)

interactive(children=(IntSlider(value=0, description='Frame', max=5459), Output()), _dom_classes=('widget-inte…

<function __main__.show_frame(frame_idx)>

## Decomposition

In [16]:
dwt_path=compute_and_save_dwt(downsampled_video_path, libpath,   device='cuda', force=True)

Loaded downsampled video data from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED13fps_downscaled.npy with shape (5460, 100, 66) and dtype bool
Loaded Gabor filter library from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\gaborLibrary_40_26_8_5_4_4.npy with shape (40, 26, 8, 5, 4, 4, 100, 66)
    Torch using: cuda, GPU index: 0, GPU name: NVIDIA GeForce RTX 4070 SUPER
    Frame batch shaped to [128, 6600] to multiply by wavelet library reshaped to torch.Size([6600, 665600]) on device cuda


Wavelet transform batched: 100%|██████████| 43/43 [00:33<00:00,  1.27it/s]


CUDA tensor memory usage:
----------------------------------------
frames_tensor       :     2.11 MB  | shape=(84, 6600)  dtype=torch.float32
l_torch_flat        : 16757.81 MB  | shape=(6600, 665600)  dtype=torch.float32
output              :   213.28 MB  | shape=(84, 665600)  dtype=torch.float32
----------------------------------------
TOTAL               : 16973.21 MB

Computed wavelet transform with shape (5460, 40, 26, 8, 5, 4, 4) 
Saved wavelet transform to D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED13fps_downscaled_lib40_26_8_5_4_4_100_66dwt.npy with shape (5460, 40, 26, 8, 5, 4, 4) and dtype float16


## Load and display decomposition

In [17]:
WT=np.load(dwt_path)
print(f"Loaded DWT from {dwt_path} with shape {WT.shape}")

downsampled_video=np.load(downsampled_video_path)

Loaded DWT from D:\SynologyDriveSyncedDATA\PROCESSED\Waven\zebra_s0_d420.0_fps59.94_RESAMPLED13fps_downscaled_lib40_26_8_5_4_4_100_66dwt.npy with shape (5460, 40, 26, 8, 5, 4, 4)


In [18]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

az_left, az_right, el_bottom, el_top = visual_coverage

param_names = ["x", "y", "angle", "size", "freq", "phase"]
param_values = [xs, ys, angles, sizes, freqs, phases]

if WT.shape[0] != downsampled_video.shape[0]:
    print(f"Warning: WT has {WT.shape[0]} frames but video has {downsampled_video.shape[0]} frames!")
n_frames = downsampled_video.shape[0]

sliders = [
    widgets.IntSlider(
        value=len(vals) // 2,
        min=0,
        max=len(vals) - 1,
        step=1,
        description=name,
        continuous_update=False,
        layout=widgets.Layout(width="250px")
    )
    for name, vals in zip(param_names, param_values)
]

time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=n_frames - 1,
    step=1,
    description="frame",
    continuous_update=False,
    layout=widgets.Layout(width="250px")
)

out_zoom = widgets.Output()
out_full = widgets.Output()
out_overlay = widgets.Output()
out_xy = widgets.Output()

def plot_view(*args):
    xi, yi, anglei, sizei, freqi, phasei = [s.value for s in sliders]
    ti = time_slider.value

    frame = downsampled_video[ti]
    filt = library[xi, yi, anglei, sizei, freqi, phasei, :, :]
    transient = WT[:, xi, yi, anglei, sizei, freqi, phasei]

    # x-y plane at current time and selected non-spatial parameters
    xy_plane = WT[ti, :, :, anglei, sizei, freqi, phasei]

    title_params = (
        f"frame={ti}, "
        f"x={xs[xi]:.2f}, y={ys[yi]:.2f}, "
        f"angle={angles[anglei]:.2f}, "
        f"size={sizes[sizei]:.2f}, "
        f"freq={freqs[freqi]:.3f}, "
        f"phase={phases[phasei]:.2f}"
    )

    # --- TOP RIGHT: temporal zoom plot
    with out_zoom:
        out_zoom.clear_output(wait=True)

        z0 = max(0, ti - 12)
        z1 = min(n_frames, ti + 13)

        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(np.arange(z0, z1), transient[z0:z1], marker="o")
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(z0, z1 - 1)
        ax.set_title("WT transient — zoom ±12 frames")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        ax.set_ylim(transient.min() , transient.max() )
        plt.tight_layout()
        plt.show()

    # --- MIDDLE: full temporal plot
    with out_full:
        out_full.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(12, 3))
        ax.plot(transient)
        ax.axvline(ti, color="red", linestyle="--", linewidth=2)
        ax.set_xlim(0, n_frames - 1)
        ax.set_title("WT transient — full")
        ax.set_xlabel("Frame")
        ax.set_ylabel("Response")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM LEFT: current video frame with Gabor overlay
    with out_overlay:
        out_overlay.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        ax.imshow(
            frame.T,
            cmap="gray",
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        filt_plot = filt.T
        v = np.max(np.abs(filt_plot))
        if v == 0:
            v = 1

        rgba = np.zeros((*filt_plot.shape, 4), dtype=float)
        pos = filt_plot > 0
        neg = filt_plot < 0

        rgba[pos, 0] = 1.0   # red = positive
        rgba[neg, 1] = 1.0   # green = negative
        rgba[..., 3] = np.abs(filt_plot) / v * 0.7

        ax.imshow(
            rgba,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        ax.set_title("Video + selected Gabor overlay")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.tight_layout()
        plt.show()

    # --- BOTTOM RIGHT: x-y WT plane
    with out_xy:
        out_xy.clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(6, 4))

        im = ax.imshow(
            xy_plane.T,
            cmap="seismic",
            vmin=-0.4,
            vmax=0.4,
            extent=[az_left, az_right, el_bottom, el_top],
            origin="lower",
            aspect="auto",
        )

        ax.scatter(xs[xi], ys[yi], c="yellow", s=60, edgecolors="black")
        ax.set_title("WT x-y plane at selected frame")
        ax.set_xlabel("Azimuth (°)")
        ax.set_ylabel("Elevation (°)")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        plt.tight_layout()
        plt.show()

    sliders[0].description = "x"
    sliders[1].description = "y"

# update on slider movement
for s in sliders + [time_slider]:
    s.observe(plot_view, names="value")

slider_box = widgets.VBox([time_slider] + sliders)

top_row = widgets.HBox([slider_box, out_zoom])
bottom_row = widgets.HBox([out_overlay, out_xy])

display(top_row, out_full, bottom_row)

plot_view()

Output()